#### Import and Load data

In [75]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df=pd.read_csv("C://Users//johny//OneDrive//Desktop//ML//churn-predictor//data//telco_churn.csv")
print(f"shape: {df.shape}")
df.head()

shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


#### Drop Useless Columns

In [76]:
#customerId is just an identifier and does not provide any useful information for the model, so we can drop it.
df =df.drop(columns=['customerID'])
print("customerId column dropped")
print(f"New shape: {df.shape}")

customerId column dropped
New shape: (7043, 20)


Fix Total Charges Column

TotalCharges was showing as text (object) instead of a number. This is because some rows have empty spaces " " instead of a real value. We need to fix this.

In [77]:
#convert TotalCharges to numeric,empty spaces become NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'],errors='coerce')

#Check how many NaN vlaues appeared in TotalCharges
print(f'Missing values in TotalCharges: {df["TotalCharges"].isnull().sum()}')

Missing values in TotalCharges: 11


#### Handling Missing Values
###### Those 11 missing rows are customers with 0 tenure (brand new customers). They have no total charges yet. Safest fix is to drop them — 11 rows out of 7043 is nothing.

In [78]:
# Drop rows where TotalCharges is missing
df =df.dropna(subset=['TotalCharges'])

print(f'New shape after dropping missing values : {df.shape}')
print(f"Any missing  values left? {df.isnull().sum().any()}")


New shape after dropping missing values : (7032, 20)
Any missing  values left? False


#### Fix Target Column(Churn)
###### Our target column Churn says "Yes" and "No" — but ML models need numbers. We convert Yes → 1, No → 0.

In [79]:
# Convert Churn to binary
df['Churn'] = df['Churn'].map({'Yes':1,'No':0})
print("Churn value counts after conversion:")
print(df['Churn'].value_counts())

Churn value counts after conversion:
Churn
0    5163
1    1869
Name: count, dtype: int64


#### Fix SeniorCitizen Column
###### SeniorCitizen is already 0 and 1 — but let's verify it's correct.

In [80]:
print(f"SeniorCitizen unique values : {df['SeniorCitizen'].unique()}")

print(f"\n SeniorCitizen type: {df['SeniorCitizen'].dtype}")

SeniorCitizen unique values : [0 1]

 SeniorCitizen type: int64


#### See all Text Columns
###### We need to know which columns are still text so we can convert them to numbers next.

In [81]:
# Get all categorical (text) columns
categorical_cols = df.select_dtypes(include='object').columns.tolist()
print(f"Text columns that need encoding: {len(categorical_cols)}")
print(categorical_cols)

Text columns that need encoding: 15
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


 #### Check Unique Values in Text Columns
 ######  Before encoding, we need to see what values are inside each column. This helps us decide HOW to encode them.

In [82]:
for col in categorical_cols:
  print(f"{col} : {df[col].unique()}")


gender : ['Female' 'Male']
Partner : ['Yes' 'No']
Dependents : ['No' 'Yes']
PhoneService : ['No' 'Yes']
MultipleLines : ['No phone service' 'No' 'Yes']
InternetService : ['DSL' 'Fiber optic' 'No']
OnlineSecurity : ['No' 'Yes' 'No internet service']
OnlineBackup : ['Yes' 'No' 'No internet service']
DeviceProtection : ['No' 'Yes' 'No internet service']
TechSupport : ['No' 'Yes' 'No internet service']
StreamingTV : ['No' 'Yes' 'No internet service']
StreamingMovies : ['No' 'Yes' 'No internet service']
Contract : ['Month-to-month' 'One year' 'Two year']
PaperlessBilling : ['Yes' 'No']
PaymentMethod : ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']


#### Encode Simple Yes/No Columns
######  Columns with only 2 values (Yes/No, Male/Female) just get converted to 0 and 1. Simple.


In [83]:
#The columns only have 2 unique values
df['gender'] = df['gender'].map({'Male':1,'Female':0})
binary_cols = ['Partner','Dependents','PhoneService','PaperlessBilling']
#map yes-no to 1-0
for col in binary_cols:
  df[col] = df[col].map({'Yes':1,'No':0})
print("Binary columns encoded!")
binary_cols = ['gender','Partner','Dependents','PhoneService','PaperlessBilling']
print(df[binary_cols].head())

Binary columns encoded!
   gender  Partner  Dependents  PhoneService  PaperlessBilling
0       0        1           0             0                 1
1       1        0           0             1                 0
2       1        0           0             1                 1
3       1        0           0             0                 0
4       0        0           0             1                 1


#### Encode Multi-Value Columns
###### These columns have 3+ unique values like Contract has "Month-to-month", "One year", "Two year". We can't just use 0/1. We use One-Hot Encoding — it creates a new column for each unique value.

In [84]:
#These columns have more than 2 unique values, so we will use one-hot encoding to convert them into dummy variables.
multi_cols = ['MultipleLines','InternetService','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies','Contract','PaymentMethod']

#one-hot encode them
df = pd.get_dummies(df,columns=multi_cols,drop_first=True)

print(f"Shape after encoding: {df.shape}")
print(f"All columns now numeric: {df.select_dtypes(include='object').shape[1]==0}")

Shape after encoding: (7032, 31)
All columns now numeric: True


#### Save clean Dataset

In [86]:
df.to_csv("C://Users//johny//OneDrive//Desktop//ML//churn-predictor//data//telco_churn_cleaned.csv",index=False)
print("Clean dataset saved!")
print(f"Final shape: {df.shape}")

Clean dataset saved!
Final shape: (7032, 31)
